# CWD Benchmark — VKG Query-Agent Eval (LLM judges + codeBased exec metrics)

Runs the **CWD (data.world ACME_Insurance)** benchmark's natural-language
questions through the deployed **ontology_query** agent (VKG path,
`QueryRuntimeArn`) and scores each answer **two ways, entirely server-side** via
AgentCore Batch Evaluations:

1. **LLM-as-Judge (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `agents/shared/eval_judges.py` (VKG family), identical to
   `5_ontology_queryagent_ac_eval.ipynb`.
2. **codeBased exec metrics (TRACE)** — `ExecutionAccuracy` (EX), `RewardVES`
   (R-VES), `SoftF1`, computed by the **BenchmarkEvalHandler** Lambda
   (`BenchmarkEvalHandlerArn`). Each is registered as a `codeBased` evaluator
   pointing at the SAME Lambda; the Lambda branches on the evaluator name. It
   extracts the agent's predicted SQL from the trace spans, reads the **gold
   SQL** from the reference input, executes both on Athena, and returns EX /
   R-VES / Soft-F1.

CWD is a **single database** (`cwd_acme`), so every row binds to the one VKG
layer `vkg-cwd_acme`.

### Prerequisites
- **`13_cwd_semantic_layer.ipynb` built the CWD VKG layer** (id `vkg-cwd_acme`).
  This notebook filters CWD rows to the db that has a completed layer and invokes
  the agent with `ontologyId = f"vkg-{row.db_id}"` (→ `vkg-cwd_acme`).
- **`BenchmarkEvalHandler` Lambda deployed** (the AgentCore stack's Part-A
  codeBased evaluator; CFN output `BenchmarkEvalHandlerArn`).
- The CWD db is ingested (manifest under `data/eval/benchmarks/cwd/`).

### Gold-packing (READ THIS — the dual-scoring contract)
Unlike BIRD (gold SQL only), CWD ships **real NL questions** — the question IS
the prompt fed to the agent. But the gold is still **SQL**, not a prose answer.
The codeBased Lambda reads gold from `expectedResponse.text` as JSON
`{"gold_sql", "db_id", "catalog"}`; the nb5 LLM judges read the expected **NL
answer** from `{assertions}` (threaded via `FINAL_ANSWER_ASSERTION_PREFIX`). These
want DIFFERENT things in the same field.

**Resolution:** we pack the gold JSON into each scenario's `expected_response`
(via `parse_multiturn_row` → `build_scenarios`, which puts it on the final
`Turn.expected_response` → the batch API's TRACE `expectedResponse.text`). That
is exactly what the codeBased Lambda reads, so **EX / R-VES / Soft-F1 are
correct** — and they are the primary CWD signal. CWD rows carry gold SQL, **not**
a prose answer, so the same packed JSON also becomes the LLM judge's
`FINAL_ANSWER_ASSERTION_PREFIX` reference — which makes `FinalAnswerFaithfulness`
**weak/uninformative for CWD** (it is comparing the agent's NL answer to a JSON
blob). That is an accepted trade-off: **the exec metrics are the primary CWD
signal**; `GoalSuccess` and `SqlGrounded` (which don't depend on a gold NL
answer) remain meaningful.


In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: huthmac+demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [4]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Load CWD rows + resolve the built VKG layers

In [5]:
# ── Load CWD rows and keep only those with a built VKG layer + gold SQL ─────
import sys
sys.path.insert(0, '..')
from pathlib import Path
from agents.shared.benchmark_adapter import load_rows
from scripts.benchmarks.ingest_common import Manifest

BENCHMARK = 'cwd'
BENCH_DIR = Path(f'../data/eval/benchmarks/{BENCHMARK}')
JSONL = BENCH_DIR / f'{BENCHMARK}.jsonl'

# Manifest maps each db_id (= namespace "cwd_<db>") -> catalog / database /
# tables. We use each row's manifest entry to build the codeBased gold payload
# (the Lambda needs the federated catalog + Athena database to execute gold SQL).
_manifest_path = None
for _cand in (BENCH_DIR / 'manifest.json', BENCH_DIR / 'deploy_manifest.json'):
    if _cand.exists():
        _manifest_path = _cand
        break
if _manifest_path is None:
    raise FileNotFoundError(
        f"No CWD manifest under {BENCH_DIR}. Run scripts/benchmarks/"
        f"deploy_datasets.py --benchmark cwd first."
    )
manifest = Manifest.load(path=_manifest_path)
_ingested_ids = {e.db_id for e in manifest.entries}

all_rows = load_rows(path=JSONL)

# ── Which CWD DBs have a COMPLETED VKG layer (built by notebook 13)? ─────────
# nb13 seeds/builds layers with the deterministic id `vkg-<namespace>` where
# namespace == the CWD db_id (e.g. "cwd_acme"). We scan DynamoDB for
# completed vkg-cwd_* layers so this notebook is self-sufficient (it does NOT
# require nb13 to have %store'd anything in this kernel).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
_meta_table = session.resource('dynamodb', region_name=region).Table(METADATA_TABLE)


def _completed_cwd_vkg_layers() -> set:
    """Return the set of db_ids (namespaces) that have a completed vkg-cwd_* layer."""
    done, scan_kw = set(), {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'vkg-cwd_', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        for it in page.get('Items', []):
            lid = it['id']                       # vkg-cwd_<db>
            if lid.startswith('vkg-'):
                done.add(lid[len('vkg-'):])      # -> cwd_<db> (== row.db_id)
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    return done


_built_db_ids = _completed_cwd_vkg_layers()
print(f"✓ Completed VKG layers for {len(_built_db_ids)} CWD db(s): "
      f"{sorted(_built_db_ids) or '(none)'}")
if not _built_db_ids:
    raise RuntimeError(
        "No completed vkg-cwd_* layers found. Run 13_cwd_semantic_layer.ipynb "
        "first (it builds one VKG layer per CWD db)."
    )

# Keep rows that are (a) ingested, (b) have a built VKG layer, and (c) carry gold
# SQL (execution-scoreable). Advisory/meta rows with no gold SQL cannot be scored.
rows = [r for r in all_rows
        if r.db_id in _ingested_ids and r.db_id in _built_db_ids and r.gold_sql]

MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    rows = rows[:MAX_ROWS]
    print(f"  ⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(rows)}-row slice (0 = all)")

if not rows:
    raise RuntimeError(
        "No scoreable CWD rows (need an ingested + VKG-built db AND gold SQL)."
    )
print(f"✓ {len(rows)} scoreable CWD row(s) across "
      f"{len({r.db_id for r in rows})} db(s)")
pd.set_option('display.max_colwidth', None)
display(pd.DataFrame([{'db_id': r.db_id, 'difficulty': r.difficulty,
                       'question': r.question} for r in rows]).head(20))


✓ Completed VKG layers for 1 CWD db(s): ['cwd_acme']
✓ 45 scoreable CWD row(s) across 1 db(s)


,db_id,difficulty,question
0,cwd_acme,,How many claims do we have?
1,cwd_acme,,What are the expense payment amount of all claims by claim number?
2,cwd_acme,,Return agents and the policy they have sold that have had a claim and the corresponding catastrophe it had.
3,cwd_acme,,"What is the total loss amounts, which is the sum of loss payment, loss reserve amount by claim number?"
4,cwd_acme,,"What are the loss payment, loss reserve, expense payment, expense reserve amount by claim number and corresponding policy number, policy holder, premium amount paid and the agent who sold it?"
5,cwd_acme,,Return all the policies and the agents that sold them by policy number and agent id
6,cwd_acme,,Return all the claims by claim number that were closed in 2019
7,cwd_acme,,What is the average time to settle a claim by policy number?
8,cwd_acme,,"What is the total loss of each claim by claim number where total loss is the sum of loss payment, loss reserve, expense payment, expense reserve amount?"
9,cwd_acme,,What is the average policy size which is the the total amount of premium divided by the number of policies?


## 3. Resolve the ontology_query runtime + the codeBased evaluator Lambda

In [6]:
# ── Look up the ontology_query runtime ARN + the BenchmarkEvalHandler Lambda ──
# BOTH come from the {PROJECT_NAME}-agentcore stack outputs (try the -dev suffix
# too). The VKG query agent's output key is `QueryRuntimeArn` (NOT
# `OntologyQueryRuntimeArn`); the codeBased exec-metrics Lambda is
# `BenchmarkEvalHandlerArn` (export `<project>-benchmark-eval-handler-arn`).
cfn = session.client('cloudformation', region_name=region)
outputs = None
for _stack in (f'{PROJECT_NAME}-agentcore', f'{PROJECT_NAME}-dev-agentcore'):
    try:
        outputs = cfn.describe_stacks(StackName=_stack)['Stacks'][0]['Outputs']
        stack_name = _stack
        break
    except Exception:  # noqa: BLE001 — try the -dev fallback stack name
        continue
if outputs is None:
    raise RuntimeError(f"AgentCore stack not found for project {PROJECT_NAME}")
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# AgentCore's batch path does NOT pass evaluatorName to the codeBased Lambda,
# so each metric is a SEPARATE function with EVAL_METRIC baked into its env.
# Resolve one ARN per metric (output keys BenchmarkEvalHandler<Metric>Arn).
BENCHMARK_EVAL_LAMBDA_ARNS = {
    'ExecutionAccuracy': output_map['BenchmarkEvalHandlerExecutionAccuracyArn'],
    'RewardVES': output_map['BenchmarkEvalHandlerRewardVESArn'],
    'SoftF1': output_map['BenchmarkEvalHandlerSoftF1Arn'],
}
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ From stack '{stack_name}':")
print(f"  Query Runtime ARN     : {_mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN)}")
print(f"  Agent ID              : {agent_id}")
for _m, _arn in BENCHMARK_EVAL_LAMBDA_ARNS.items():
    print(f"  BenchmarkEval {_m:<17}: {_mask_acct(_arn)}")


✓ From stack 'semantic-layer-dev-agentcore':
  Query Runtime ARN     : arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_ontology_query-HKGVpkE6XK
  Agent ID              : semantic_layer_dev_ontology_query-HKGVpkE6XK
  BenchmarkEval ExecutionAccuracy: arn:aws:lambda:us-east-1:********4087:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerExec-LJmXMjCZFupT
  BenchmarkEval RewardVES        : arn:aws:lambda:us-east-1:********4087:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerRewa-q5QW4gXFaM0O
  BenchmarkEval SoftF1           : arn:aws:lambda:us-east-1:********4087:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerSoft-6YJKDVuQkczd


## 4. Create the evaluators — LLM judges (SESSION) + codeBased exec metrics (TRACE)

Two evaluator families, both scored server-side:

| Evaluator | Kind | Level | Source |
|:--|:--|:--|:--|
| `GoalSuccess` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `FinalAnswerFaithfulness` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `SqlGrounded` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `cwd_ExecutionAccuracy` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `cwd_RewardVES` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `cwd_SoftF1` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |

> The codeBased evaluators branch on the evaluator NAME (the Lambda matches the
> canonical substrings `ExecutionAccuracy` / `RewardVES` / `SoftF1`), so a
> `cwd_` prefix is fine. All three point at the SAME Lambda ARN.

> **`Builtin.GoalSuccessRate` is intentionally omitted** — nb5 documents that the
> AWS-managed builtin mis-scores this deterministic-graph agent (it mistakes an
> intermediate intent/SPARQL span for the answer). The custom `GoalSuccess`
> SESSION judge replaces it. (If you want the builtin as a cross-check, append
> `'Builtin.GoalSuccessRate'` to `ALL_EVALUATOR_IDS` in the next cell — read it
> skeptically.)


In [7]:
# ── LLM-as-Judge (SESSION) evaluators — VKG family, from eval_judges ─────────
try:
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID

_cp = session.client('bedrock-agentcore-control', region_name=region)

# family='vkg' → [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded] (byte-identical
# to nb5's judges). GoalSuccess is the custom replacement for the un-editable
# Builtin.GoalSuccessRate on this deterministic-graph agent.
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = create_custom_judges(
    control_client=_cp, family='vkg',
)
CUSTOM_JUDGE_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print("✓ Custom SESSION LLM-as-Judge evaluators created:")
print(f"  GoalSuccess             : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             : {SQL_GROUNDED_ID}")


# ── codeBased exec-metric (TRACE) evaluators — one per metric, same Lambda ───
# The AgentCore CreateEvaluator codeBased config is
#   {'codeBased': {'lambdaConfig': {'lambdaArn': <arn>, 'lambdaTimeoutInSeconds': N}}}
# level=TRACE (one score per trace). We register three, whose names carry the
# canonical metric substrings the Lambda branches on (ExecutionAccuracy / RewardVES
# / SoftF1). A fresh suffix keeps redeploys unique.
import uuid as _uuid
_SUFFIX = _uuid.uuid4().hex[:8]
# Lambda run_sql polls up to 120s PER query and runs gold THEN predicted, so give
# the codeBased invocation headroom (max allowed is 300s).
_LAMBDA_TIMEOUT_S = 280


def _create_codebased_evaluator(*, metric_name: str) -> str:
    """Create one codeBased exec-metric evaluator (TRACE) and return its id.

    Parameters:
        metric_name: The metric substring the Lambda branches on — one of
            'ExecutionAccuracy', 'RewardVES', 'SoftF1'.
    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f'cwd_{metric_name}_{_SUFFIX}',
        level='TRACE',
        evaluatorConfig={
            'codeBased': {
                'lambdaConfig': {
                    # Per-metric function ARN — EVAL_METRIC in its env
                    # tells the shared handler which metric to score.
                    'lambdaArn': BENCHMARK_EVAL_LAMBDA_ARNS[metric_name],
                    'lambdaTimeoutInSeconds': _LAMBDA_TIMEOUT_S,
                }
            }
        },
    )
    return resp['evaluatorId']


EX_ID = _create_codebased_evaluator(metric_name='ExecutionAccuracy')
RVES_ID = _create_codebased_evaluator(metric_name='RewardVES')
SOFTF1_ID = _create_codebased_evaluator(metric_name='SoftF1')
CODEBASED_IDS = [EX_ID, RVES_ID, SOFTF1_ID]
print("\n✓ codeBased exec-metric evaluators created (TRACE, same Lambda):")
print(f"  cwd_ExecutionAccuracy : {EX_ID}")
print(f"  cwd_RewardVES         : {RVES_ID}")
print(f"  cwd_SoftF1            : {SOFTF1_ID}")

# All evaluator ids the batch job scores. LLM judges (SESSION) + exec metrics
# (TRACE) — the batch API applies each at its own level from one flat list.
ALL_EVALUATOR_IDS = [*CUSTOM_JUDGE_IDS, *CODEBASED_IDS]

# Display-name map for the results tables (event names drop the trailing -<hash>).
_EVAL_NAME = {
    GOAL_SUCCESS_ID: 'GoalSuccess', ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
    SQL_GROUNDED_ID: 'SqlGrounded', EX_ID: 'ExecutionAccuracy',
    RVES_ID: 'RewardVES', SOFTF1_ID: 'SoftF1',
}
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})


✓ Custom SESSION LLM-as-Judge evaluators created:
  GoalSuccess             : GoalSuccess_ce88ee75-nXcApVFCCr
  FinalAnswerFaithfulness : FinalAnswerFaithfulness_ce88ee75-xU1MqJFveP
  SqlGrounded             : SqlGrounded_ce88ee75-1jYLysFieo



✓ codeBased exec-metric evaluators created (TRACE, same Lambda):
  cwd_ExecutionAccuracy : cwd_ExecutionAccuracy_c179ca87-B8Mhl9HKGH
  cwd_RewardVES         : cwd_RewardVES_c179ca87-8sIHdz2ell
  cwd_SoftF1            : cwd_SoftF1_c179ca87-lz7qmcFENX


## 5. Build the Dataset (one scenario per CWD row) + agent_invoker

Each CWD row is one `PredefinedScenario` (its own session), bound to its DB's
VKG layer via `ontologyId = f"vkg-{row.db_id}"`.

**Gold-packing:** each row's `expected_response` is set to
`json.dumps({"gold_sql", "db_id", "catalog"})` (via `parse_multiturn_row` +
`build_scenarios`). The batch API delivers that as the TRACE
`expectedResponse.text` — which the codeBased Lambda reads for EX / R-VES /
Soft-F1. (It also becomes the LLM judge's `FINAL_ANSWER_ASSERTION_PREFIX`
reference; see the intro for why `FinalAnswerFaithfulness` is weak on CWD.)


In [8]:
# Per-row chat-stream invoker + dataset builder (mirrors nb5, but binds ontologyId
# PER ROW and packs the codeBased gold into expected_response).
import time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# ── Pack the codeBased gold into each row's Expected_Answer ──────────────────
# The codeBased Lambda reads expectedResponse.text = json.dumps({gold_sql, db_id,
# catalog}); build_scenarios threads a row's Expected_Answer onto the final
# Turn.expected_response (→ the TRACE expectedResponse.text). So we set
# Expected_Answer to the packed gold. db_id (= namespace cwd_<db>) is the Athena
# database the Lambda executes gold SQL in; catalog is the federated S3 Tables
# catalog (per-row from the manifest entry). One scenario per row.
_specs = []
SCENARIO_LAYER = {}        # scenario_id -> ontologyId (vkg-<db_id>)
SCENARIO_ROW = {}          # scenario_id -> the BenchmarkRow (for the results file)
for i, row in enumerate(rows):
    entry = manifest.entry(db_id=row.db_id)
    gold_payload = json.dumps({
        'gold_sql': row.gold_sql,
        'db_id': row.db_id,          # == namespace cwd_<db> == Athena database
        'catalog': entry.catalog,    # federated S3 Tables catalog id
    })
    spec = parse_multiturn_row(
        {'Natural_Language_Question': row.question, 'Expected_Answer': gold_payload},
        index=i,
    )
    layer_id = f'vkg-{row.db_id}'    # SAME id nb13 built
    SCENARIO_LAYER[spec['scenario_id']] = layer_id
    SCENARIO_ROW[spec['scenario_id']] = row
    _specs.append(spec)

# build_scenarios takes ONE ontology_id (stamped into scenario metadata), but CWD
# rows span DBs, so call it PER spec with that row's layer id and concatenate. The
# real ontologyId binding happens in agent_invoker (below) via build_chat_payload.
scenarios = []
for spec in _specs:
    scenarios.extend(build_scenarios(
        [spec], ontology_id=SCENARIO_LAYER[spec['scenario_id']],
        simulated_enabled=False))
dataset = Dataset(scenarios=scenarios)
print(f"✓ Dataset: {len(dataset.scenarios)} CWD scenario(s) (one per row)")


AGENT_RUNS = {}            # keyed by run_key(session_id, turn_idx)

# Hard wall-clock cap per agent invocation. The 2026-11-05 run had 12 rows hang
# for 2-8h each (agent loop ran to the model max_tokens limit while the runtime
# held the HTTP stream open until its 8h session ceiling); urlopen(timeout=900)
# is only a per-read socket timeout and never fired. Run the call on a worker
# thread and give up after MAX_ROW_WALL_S so a hung row fails fast and is
# recorded as an invoke_error instead of stalling the whole batch.
import concurrent.futures as _cf
MAX_ROW_WALL_S = int(os.environ.get("MAX_ROW_WALL_S", "1800"))
_INVOKE_POOL = _cf.ThreadPoolExecutor(max_workers=16, thread_name_prefix="cwd-invoke")


def _invoke_runtime_with_deadline(arn: str, session_id: str, payload: bytes) -> bytes:
    """Call _invoke_runtime but raise TimeoutError after MAX_ROW_WALL_S seconds."""
    fut = _INVOKE_POOL.submit(_invoke_runtime, arn, session_id, payload)
    try:
        return fut.result(timeout=MAX_ROW_WALL_S)
    except _cf.TimeoutError:
        raise TimeoutError(f"agent run exceeded MAX_ROW_WALL_S={MAX_ROW_WALL_S}s") from None


def _scenario_id_for_session(sess: str) -> str:
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Invoke the VKG query agent for one CWD row via the chat-stream path.

    Binds ontologyId to the row's per-DB VKG layer (vkg-<db_id>), parses the SSE
    body, and records the predicted SQL + cost/latency into AGENT_RUNS.
    """
    sid = invoker_input.session_id
    scenario_id = _scenario_id_for_session(sid)
    ontology_id = SCENARIO_LAYER.get(scenario_id, '')
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=ontology_id, turn_idx=turn_idx,
                                 user_id="cwd-eval", source="eval")
    payload["mode"] = "vkg"                    # VKG path (default is semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime_with_deadline(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                                            json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001 — one row failure must not abort the batch
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "scenario_id": scenario_id, "turn_idx": turn_idx,
        "ontology_id": ontology_id, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "executed_sql": parsed.get("executed_sql", ""), "degraded": parsed.get("degraded"),
        "rows": parsed.get("rows", []) or [], "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "answer" if parsed.get("sql") else ("clarify" if parsed.get("clarification") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{scenario_id} {ontology_id}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))


✓ Dataset: 45 CWD scenario(s) (one per row)


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent once per scenario (via `agent_invoker`),
waits `ingestion_delay_seconds` for CloudWatch span ingestion, submits ONE
`StartBatchEvaluation` over all six evaluators (SESSION judges + TRACE exec
metrics), and polls to completion. All scoring is in-service.


In [9]:
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")
print(f"Evaluators   : {ALL_EVALUATOR_IDS}")
print(f"Scenarios    : {len(dataset.scenarios)}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=420,   # give spans (incl. the SQL-exec span) time to land
)

batch_runner = BatchEvaluationRunner(region=region)

cfg = BatchEvaluationRunConfig(
    batch_evaluation_name=f"cwd_vkg_bench_{uuid.uuid4().hex[:8]}",
    description="CWD VKG query-agent eval: LLM judges (SESSION) + codeBased "
                "exec metrics EX/R-VES/Soft-F1 (TRACE).",
    evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
    data_source=batch_data_source,
    max_concurrent_scenarios=3,    # synchronous agent, up to 900s/row
    polling_timeout_seconds=5400,  # VKG rows are multi-phase + the codeBased Lambda runs 2 Athena queries/row
    polling_interval_seconds=30,
)

print(f"\n=== {cfg.batch_evaluation_name} ===")
print("Invoking the agent per row, then evaluating server-side...")
batch_result = batch_runner.run_dataset_evaluation(
    config=cfg, dataset=dataset, agent_invoker=agent_invoker)
print(f"\n✓ Batch status: {batch_result.status}")
if getattr(batch_result, 'agent_invocation_failures', None):
    print(f"⚠ Agent invocation failures: {len(batch_result.agent_invocation_failures)}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")


SERVICE_NAME : semantic_layer_dev_ontology_query-HKGVpkE6XK.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology_query-HKGVpkE6XK-DEFAULT
Evaluators   : ['GoalSuccess_ce88ee75-nXcApVFCCr', 'FinalAnswerFaithfulness_ce88ee75-xU1MqJFveP', 'SqlGrounded_ce88ee75-1jYLysFieo', 'cwd_ExecutionAccuracy_c179ca87-B8Mhl9HKGH', 'cwd_RewardVES_c179ca87-8sIHdz2ell', 'cwd_SoftF1_c179ca87-lz7qmcFENX']
Scenarios    : 45



=== cwd_vkg_bench_45c878df ===
Invoking the agent per row, then evaluating server-side...


  OK [gt-row-02 vkg-cwd_acme] 21.6s . none


  OK [gt-row-00 vkg-cwd_acme] 32.77s . answer


  OK [gt-row-03 vkg-cwd_acme] 23.03s . none


  OK [gt-row-04 vkg-cwd_acme] 19.83s . none


... (38 outputs left out)

  OK [gt-row-44 vkg-cwd_acme] 17.66s . none


  OK [gt-row-41 vkg-cwd_acme] 59.8s . answer


  OK [gt-row-43 vkg-cwd_acme] 26.97s . none



✓ Batch status: COMPLETED_WITH_ERRORS
⚠ Error details: ['17 of 45 sessions failed during batch evaluation.']


## 7. Results — LLM-judge aggregates + exec-metric (EX / R-VES / Soft-F1) aggregates

1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results`.
   The three codeBased evaluators' `average_score` **are** the mean EX / R-VES /
   Soft-F1 over the scored rows; the three judges' are the mean pass rates.
2. **Per-row evaluator scores** — `fetch_evaluation_events()`, joined by session id.
3. **Per-row agent detail** — predicted SQL / cost / latency from `AGENT_RUNS`.


In [10]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores (LLM judges + exec metrics) ────────────
agg_rows = []
_agg_by_name = {}
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (stats.average_score if stats and stats.average_score is not None else None)
        name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
        _agg_by_name[name] = round(avg, 4) if avg is not None else None
        agg_rows.append({
            'Evaluator': name,
            'AvgScore': f"{avg:.3f}" if avg is not None else None,
            'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

# ── 2. Per-row evaluator events ──────────────────────────────────────────────
def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-row events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    sid = _scenario_id_for_session(sess)
    _row = SCENARIO_ROW.get(sid)
    event_rows.append({
        'scenario_id': sid,
        'db_id': getattr(_row, 'db_id', ''),
        'difficulty': getattr(_row, 'difficulty', '') or '',
        'question': getattr(_row, 'question', ''),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': str(_event_field(e, 'gen_ai.evaluation.explanation') or ''),
        'error_type': _event_field(e, 'error.type'),
        'error_message': str(_event_field(e, 'error.message') or '')[:300],
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-row evaluator events: {len(df_events)}")

# ── Per-difficulty EX/R-VES/Soft-F1 from the exec-metric events ──────────────
_EXEC_NAMES = {'ExecutionAccuracy', 'RewardVES', 'SoftF1'}
by_diff = {}
for r in event_rows:
    if r['evaluator'] not in _EXEC_NAMES or not isinstance(r['score'], (int, float)):
        continue
    by_diff.setdefault(r['difficulty'] or 'unknown', {}).setdefault(
        r['evaluator'], []).append(r['score'])
exec_by_difficulty = {
    band: {m: round(sum(v) / len(v), 4) for m, v in metrics.items()}
    for band, metrics in by_diff.items()
}

# ── Headline exec metrics from the per-row events ────────────────────────────
# The service's evaluatorSummaries carry NO statistics for codeBased evaluators
# (observed 2026-10-02 and 2026-11-06: statistics == {} even with 22+ scored
# rows), so _agg_by_name is None for EX/R-VES/Soft-F1. Compute the headline the
# same way the BIRD per-DB driver does: mean over rows that produced a numeric
# score, and report coverage (scored / total rows) alongside so a low-n figure
# cannot pass as a full-benchmark number.
exec_scores = {m: [] for m in _EXEC_NAMES}
for r in event_rows:
    if r['evaluator'] in _EXEC_NAMES and isinstance(r['score'], (int, float)):
        exec_scores[r['evaluator']].append(r['score'])
exec_metric_aggregates = {
    m: (round(sum(v) / len(v), 4) if v else _agg_by_name.get(m))
    for m, v in exec_scores.items()
}
exec_metric_coverage = {
    m: {'scored': len(v), 'total_rows': len(rows),
        'coverage': round(len(v) / len(rows), 4) if rows else None}
    for m, v in exec_scores.items()
}

# ── Failure-reason breakdown (why rows were NOT scored) ──────────────────────
# evaluator -> error.type -> count. Separates dataset defects (GoldExecutionFailed:
# gold SQL cannot run on Athena) from agent faults (SpanStatusErrorException: the
# agent trace has an error-status span so the LLM judges refuse to score) and
# from telemetry loss.
failure_breakdown = {}
for r in event_rows:
    if r.get('error_type'):
        failure_breakdown.setdefault(r['evaluator'], {}).setdefault(r['error_type'], 0)
        failure_breakdown[r['evaluator']][r['error_type']] += 1

# ── 3. Per-row agent detail (predicted SQL / cost / latency) ─────────────────
agent_detail = []
for rec in AGENT_RUNS.values():
    agent_detail.append({
        'scenario_id': rec['scenario_id'], 'ontology_id': rec['ontology_id'],
        'question': rec['message'], 'agent_sql': rec['agent_sql'],
        'executed_sql': rec['executed_sql'], 'degraded': rec['degraded'],
        'wall_clock_s': rec['wall_clock_s'], 'error': rec['invoke_error'],
    })
df_agent = pd.DataFrame(agent_detail)

# ── Persist ──────────────────────────────────────────────────────────────────
combined_file = f"../data/eval/results/cwd_vkg_benchmark_eval_{timestamp}.json"
combined = {
    'benchmark': 'cwd',
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN,
    'benchmark_eval_lambda_arns': BENCHMARK_EVAL_LAMBDA_ARNS,
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'llm_judges': {'GoalSuccess': GOAL_SUCCESS_ID,
                   'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                   'SqlGrounded': SQL_GROUNDED_ID},
    'exec_metric_evaluators': {'ExecutionAccuracy': EX_ID, 'RewardVES': RVES_ID,
                               'SoftF1': SOFTF1_ID},
    # Headline: the LLM-judge aggregates AND the exec-metric (EX/R-VES/Soft-F1) aggregates.
    'llm_judge_aggregates': {k: _agg_by_name.get(k) for k in
                             ('GoalSuccess', 'FinalAnswerFaithfulness', 'SqlGrounded')},
    'exec_metric_aggregates': exec_metric_aggregates,
    'exec_metric_coverage': exec_metric_coverage,
    'failure_breakdown': failure_breakdown,
    'agent_invoke_failures': sum(1 for a in agent_detail if a.get('error')),
    'max_row_wall_s': MAX_ROW_WALL_S,
    'exec_metrics_by_difficulty': exec_by_difficulty,
    'aggregate_summaries': agg_rows,
    'per_row_events': event_rows,
    'agent_detail': agent_detail,
    'n_rows': len(rows),
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
_ex = combined['exec_metric_aggregates']
_cov = combined['exec_metric_coverage']
print(f"\nHeadline exec metrics (rows={len(rows)}; mean over SCORED rows, coverage = scored/rows):")
for _m in ('ExecutionAccuracy', 'RewardVES', 'SoftF1'):
    print(f"  {_m:<18}: {_ex.get(_m)}  (n={_cov[_m]['scored']}/{len(rows)}, coverage={_cov[_m]['coverage']})")
print(f"\nAgent invoke failures (timeout / transport): {combined['agent_invoke_failures']} of {len(rows)}")
if failure_breakdown:
    print("\nUnscored-row reasons (evaluator -> error.type -> rows):")
    for _evn in sorted(failure_breakdown):
        print(f"  {_evn:<24}: {failure_breakdown[_evn]}")
if exec_by_difficulty:
    print("\nExec metrics by difficulty:")
    for band in sorted(exec_by_difficulty):
        m = exec_by_difficulty[band]
        print(f"  {band:>12}: EX={m.get('ExecutionAccuracy')}  "
              f"R-VES={m.get('RewardVES')}  Soft-F1={m.get('SoftF1')}")
print("\n=== Per-row evaluator scores ===")
display(df_events)
print("=== Per-row agent detail ===")
display(df_agent)


Batch ID : cwd_vkg_bench_45c878df-94cc14dd1d
Status   : COMPLETED_WITH_ERRORS

Sessions: completed=28 failed=17 total=45



Per-row evaluator events: 270

✓ Wrote ../data/eval/results/cwd_vkg_benchmark_eval_20260727_173922.json

=== Aggregate per-evaluator scores ===


,Evaluator,AvgScore,Evaluated,Failed
0,SoftF1,NaN,28,17
1,RewardVES,NaN,28,17
2,GoalSuccess,0.000,45,0
3,SqlGrounded,1.000,45,0
4,FinalAnswerFaithfulness,0.000,45,0
5,ExecutionAccuracy,NaN,28,17



Headline exec metrics (n=45): EX=None  R-VES=None  Soft-F1=None

Exec metrics by difficulty:
       unknown: EX=0.0  R-VES=0.0  Soft-F1=0.0

=== Per-row evaluator scores ===


,scenario_id,db_id,difficulty,question,evaluator,score,label,explanation
0,gt-row-19,cwd_acme,,What is the total amount of premiums that a policy holder has paid by policy number?,ExecutionAccuracy,NaN,NaN,
1,gt-row-19,cwd_acme,,What is the total amount of premiums that a policy holder has paid by policy number?,SoftF1,NaN,NaN,
2,gt-row-19,cwd_acme,,What is the total amount of premiums that a policy holder has paid by policy number?,SqlGrounded,1.0,pass,"The agent's final answer states that the query failed to execute against the data, and there is no executed SQL present in the context. No SQL was run on Athena, so there is no schema grounding to verify. Per the instructions, when there is no executed SQL (the agent degraded rather than running ungrounded SQL), the grounding invariant is considered upheld."
3,gt-row-19,cwd_acme,,What is the total amount of premiums that a policy holder has paid by policy number?,RewardVES,NaN,NaN,
4,gt-row-19,cwd_acme,,What is the total amount of premiums that a policy holder has paid by policy number?,GoalSuccess,0.0,fail,"The final-answer record's OUTPUT states that the query failed to execute and no result was provided to the user. The expected gold answer required a substantive result (total premiums by policy number). Since the agent never delivered a factual answer matching the gold SQL's result, the assertion 'the conversation's final answer matches the gold result' is violated. This is a failure to reach a substantive answer when one was required."
...,...,...,...,...,...,...,...,...
265,gt-row-06,cwd_acme,,Return all the claims by claim number that were closed in 2019,SoftF1,0.0,FAIL,No predicted SQL found in the trace spans; scored 0.
266,gt-row-06,cwd_acme,,Return all the claims by claim number that were closed in 2019,FinalAnswerFaithfulness,0.0,fail,"The final-answer record states that the required data (Claim, claim_number, status_cd, closed_date) isn't available in the semantic layer, and thus the question cannot be answered. However, the gold SQL shows that the data actually exists (claim table with company_claim_number and claim_close_date), and the expected answer is a list of claim numbers closed in 2019. The agent's response fails to provide this substantive answer, instead giving a degraded 'data not available' explanation that contradicts the expected outcome. This does not match the expected final answer."
267,gt-row-06,cwd_acme,,Return all the claims by claim number that were closed in 2019,RewardVES,0.0,FAIL,No predicted SQL found in the trace spans; scored 0.
268,gt-row-06,cwd_acme,,Return all the claims by claim number that were closed in 2019,SqlGrounded,1.0,pass,"The agent determined the ontology slice was insufficient (missing Claim, Claim.claim_number, Claim.status_cd, Claim.closed_date) and refused to execute any SQL, returning a degraded response instead. There is no executed SQL present in the context, so the grounding invariant is upheld by default per the instructions."


=== Per-row agent detail ===


,scenario_id,ontology_id,question,agent_sql,executed_sql,degraded,wall_clock_s,error
0,gt-row-02,vkg-cwd_acme,Return agents and the policy they have sold that have had a claim and the corresponding catastrophe it had.,,,None,21.60,None
1,gt-row-00,vkg-cwd_acme,How many claims do we have?,SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE {\n ?c a <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/Claim> .\n},SELECT 1 AS uselessVariable\nFROM (SELECT 1 ) tdummy\n,None,32.77,None
2,gt-row-03,vkg-cwd_acme,"What is the total loss amounts, which is the sum of loss payment, loss reserve amount by claim number?",,,None,23.03,None
3,gt-row-04,vkg-cwd_acme,"What are the loss payment, loss reserve, expense payment, expense reserve amount by claim number and corresponding policy number, policy holder, premium amount paid and the agent who sold it?",,,None,19.83,None
4,gt-row-05,vkg-cwd_acme,Return all the policies and the agents that sold them by policy number and agent id,,,None,18.79,None
5,gt-row-06,vkg-cwd_acme,Return all the claims by claim number that were closed in 2019,,,None,21.90,None
6,gt-row-08,vkg-cwd_acme,"What is the total loss of each claim by claim number where total loss is the sum of loss payment, loss reserve, expense payment, expense reserve amount?",,,None,20.48,None
7,gt-row-01,vkg-cwd_acme,What are the expense payment amount of all claims by claim number?,SELECT ?claim_id (SUM(?amt) AS ?totalExpenseAmount)\nWHERE {\n ?er a <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ExpenseReserve> ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ExpenseReserve/claim_id> ?claim_id ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ExpenseReserve/reserve_amount> ?amt .\n}\nGROUP BY ?claim_id\nORDER BY ?claim_id,"SELECT R1.CLAIM_ID AS ""CLAIM_ID1m14"", SUM(CAST(R1.""RESERVE_AMOUNT"" AS DOUBLE PRECISION)) AS ""sum1""\nFROM CWD_ACME.EXPENSE_RESERVE R1\nWHERE (R1.""RESERVE_AMOUNT"" IS NOT NULL AND R1.CLAIM_ID IS NOT NULL)\nGROUP BY R1.CLAIM_ID\nORDER BY R1.CLAIM_ID NULLS FIRST",None,111.37,None
8,gt-row-10,vkg-cwd_acme,What are all our policies that have a claim associated to them by policy and claim number?,SELECT DISTINCT ?policy_id ?claim_id WHERE {\n ?cc a <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ClaimCoverage> ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ClaimCoverage/claim_id> ?claim_id ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/ClaimCoverage/hasPolicyCoverageDetail> ?pcd .\n ?prem a <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/Premium> ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/Premium/hasPolicyCoverageDetail> ?pcd ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/Premium/policy_id> ?policy_id .\n}\nORDER BY ?policy_id ?claim_id,"SELECT R1.CLAIM_ID AS ""CLAIM_ID1m1"", R1.COVERAGE_ID AS ""COVERAGE_ID1m10"", R1.POLICY_ID AS ""POLICY_ID1m51""\nFROM CWD_ACME.CLAIM_COVERAGE R1, CWD_ACME.PREMIUM R2\nWHERE (R1.POLICY_ID IS NOT NULL AND R1.CLAIM_ID IS NOT NULL AND R1.COVERAGE_ID = R2.COVERAGE_ID)\nORDER BY R1.POLICY_ID NULLS FIRST, R1.CLAIM_ID NULLS FIRST",None,67.33,None
9,gt-row-07,vkg-cwd_acme,What is the average time to settle a claim by policy number?,"SELECT ?claim_id (AVG(?days) AS ?avgDaysToSettle)\nWHERE {\n ?lr a <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/LossReserve> ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/LossReserve/claim_id> ?claim_id ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/LossReserve/reserve_status_cd> ?status ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/LossReserve/set_date> ?setDate ;\n <http://cwd-cwd_acme-ontology-20260723-170827/ontology/vkg-cwd_acme/LossReserve/status_date> ?statusDate .\n FILTER(?status = ""Closed"")\n BIND((?statusDate - ?setDate) AS ?days)\n}\nGROUP BY ?c

## Summary

This notebook runs the **CWD (ACME_Insurance)** benchmark through the deployed
**ontology_query** (VKG) agent and scores each answer **server-side, two ways**:

1. **LLM judges (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `eval_judges` (VKG family), identical to nb5.
2. **codeBased exec metrics (TRACE)** — `cwd_ExecutionAccuracy` (EX),
   `cwd_RewardVES` (R-VES), `cwd_SoftF1`, three `codeBased` evaluators pointing
   at the `BenchmarkEvalHandler` Lambda (`BenchmarkEvalHandlerArn`), which extracts
   the predicted SQL from the trace, reads the packed gold from
   `expectedResponse.text`, and executes both on Athena.

### Gold-packing / dual-scoring trade-off
CWD ships **real NL questions** (the question is the prompt), but the gold is
still SQL, not prose. The gold JSON `{gold_sql, db_id, catalog}` is packed into
each scenario's `expected_response` (→ TRACE `expectedResponse.text`), which is
what the codeBased Lambda reads — so **EX / R-VES / Soft-F1 are the correct,
primary CWD signal**. Because there is no prose gold answer, that same packed JSON
is what the LLM `FinalAnswerFaithfulness` judge sees as its expected answer →
**weak for CWD** (accepted). `GoalSuccess` and `SqlGrounded` don't depend on a
gold NL answer and remain meaningful.

### Layer binding
`ontologyId = f"vkg-{row.db_id}"` — every CWD row's `db_id` is `cwd_acme`, so this
resolves to `vkg-cwd_acme`, the exact id `13_cwd_semantic_layer.ipynb` built. Rows
are filtered to the db with a completed `vkg-cwd_*` layer.

### Notes
- **R-VES caveat**: the Lambda times both gold and predicted SQL on Athena;
  there is no external reference timing for CWD, so read R-VES as a within-arm
  relative signal. **EX** and **Soft-F1** are the portable metrics.
- Results → `../data/eval/results/cwd_vkg_benchmark_eval_*.json` (LLM-judge
  aggregates + exec-metric aggregates + per-difficulty EX/R-VES/Soft-F1 +
  per-row detail).
